In [ ]:
import os
import time
import subprocess
import chromadb
from google import genai
from google.genai import types
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage

# ---------------------------------------------------------
# 1. Configuration & Setup
# ---------------------------------------------------------
# Replace with your actual Google API key
os.environ["GEMINI_API_KEY"] = "" # input your API key here
client = genai.Client()

# Initialize ChromaDB
DB_PATH = "./cctv_chroma_db"
chroma_client = chromadb.PersistentClient(path=DB_PATH)
# We use a new collection name to avoid mixing with any old text-based vectors
collection = chroma_client.get_or_create_collection(name="direct_video_vectors")
dimension = 768

# ---------------------------------------------------------
# 2. FFmpeg Video Utilities
# ---------------------------------------------------------
def get_video_duration(video_path):
    """Uses FFprobe to get the exact duration of the raw video."""
    cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", 
           "-of", "default=noprint_wrappers=1:nokey=1", video_path]
    return float(subprocess.check_output(cmd).decode('utf-8').strip())

def extract_video_clip(input_path, start_sec, duration, output_path):
    """Losslessly slices a specific time window from the raw video."""
    cmd = [
        "ffmpeg", "-y", "-ss", str(start_sec), "-i", input_path, 
        "-t", str(duration), "-c", "copy", "-avoid_negative_ts", "make_zero", output_path
    ]
    subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    return output_path if os.path.exists(output_path) else None

# ---------------------------------------------------------
# 3. PHASE 1: Direct Multimodal Ingestion (Run Once)
# ---------------------------------------------------------

def ingest_raw_video_direct(video_path, chunk_duration=15.0):
    """
    Directly embeds raw video chunks into ChromaDB using INLINE DATA,
    bypassing the Google Cloud File API entirely.
    """
    video_id = os.path.splitext(os.path.basename(video_path))[0]
    duration = get_video_duration(video_path)
    
    print(f"\n=== 🎬 STARTING DIRECT MULTIMODAL INGESTION: {video_id} ({duration:.1f}s) ===")
    
    current_sec = 0.0
    chunk_idx = 0
    
    while current_sec < duration:
        end_sec = min(current_sec + chunk_duration, duration)
        clip_path = f"temp_ingest_{chunk_idx}.mp4"
        
        # 1. Extract 15-second Chunk
        extract_video_clip(video_path, current_sec, chunk_duration, clip_path)
        print(f"\n⚙️ Processing Chunk {chunk_idx} [{current_sec:.1f}s - {end_sec:.1f}s]...")
        
        try:
            # 2. READ VIDEO AS RAW BYTES (Bypasses the buggy File API)
            with open(clip_path, "rb") as f:
                video_bytes = f.read()
            
            # 3. DIRECT EMBEDDING (Inline Bytes -> Math)
            embed_response = client.models.embed_content(
                model='gemini-embedding-2-preview',
                # Pass the raw bytes directly to the model
                contents=types.Part.from_bytes(data=video_bytes, mime_type="video/mp4"),
                config=types.EmbedContentConfig(
                    task_type="RETRIEVAL_DOCUMENT",
                    output_dimensionality=dimension
                )
            )
            vector = embed_response.embeddings[0].values
            
            # 4. Save Vector to ChromaDB 
            collection.add(
                embeddings=[vector],
                documents=[""], # No text!
                metadatas=[{"video_id": video_id, "start_sec": current_sec, "end_sec": end_sec}],
                ids=[f"{video_id}_chunk_{chunk_idx}"]
            )
            print("   ✅ Video chunk embedded and saved.")
            
        except Exception as e:
            print(f"   ❌ Error processing chunk {chunk_idx}: {e}")
            
        # Clean up local file
        if os.path.exists(clip_path):
            os.remove(clip_path)
            
        current_sec += chunk_duration
        chunk_idx += 1
        
    print(f"=== 🎉 DIRECT INGESTION COMPLETE! '{video_id}' is now searchable. ===")
# ---------------------------------------------------------
# 4. PHASE 2: Chat & Retrieval (Run Anytime)
# ---------------------------------------------------------


# ---------------------------------------------------------
# 4. PHASE 2: Chat & Retrieval (LLM CONTEXT PRUNING)
# ---------------------------------------------------------
def chat_with_raw_video_direct(user_query, raw_video_path, chat_history=None):
    """
    Retrieves top matches, groups them into isolated chronological clusters, 
    extracts clips, and synthesizes an answer using dynamic LLM memory management.
    """
    if chat_history is None:
        chat_history = []

    video_id = os.path.splitext(os.path.basename(raw_video_path))[0]
    print(f"\n=== 🕵️‍♂️ CCTV ASSISTANT QUERY ===\nQuestion: '{user_query}'")

    # --- Step 1: LLM-Driven Conversational Query Rewriting (CQR) ---
    print("🦙 Asking Gemma 4 to resolve context and rewrite the query...")
    
    # Format the entire history into a readable string
    full_history_text = ""
    for idx, turn in enumerate(chat_history):
        full_history_text += f"[Turn {idx+1}] User: {turn['question']} | AI: {turn['answer']}\n"
    
    system_prompt = f"""You are an elite intent parser for a surveillance system.
Your job is to read the user's CURRENT QUERY and the FULL CHAT HISTORY, and output two things separated by a '|' character:
1. RELEVANT HISTORY: A brief summary of ONLY the facts from the history needed to understand the current query. If no history is needed, write 'NONE'.
2. REFINED QUERY: Rewrite the user's current query into a single, fully self-contained natural language sentence. Resolve any pronouns (e.g., replace "he" with "the man in the red shirt") and include necessary context from previous turns so the query makes perfect sense on its own. DO NOT output comma-separated keywords. Write it exactly as a human would ask a complete, standalone question.

FULL CHAT HISTORY:
{full_history_text if full_history_text else "No previous history."}

CURRENT QUERY: {user_query}

Respond strictly in this format: RELEVANT HISTORY | REFINED QUERY"""

    try:
        #local_llm = ChatOllama(model="llama3")
        #ai_message = local_llm.invoke([HumanMessage(content=system_prompt)])

        ai_message = client.models.generate_content(
            model='gemma-4-9b-it', # Ensure you use the '-it' (Instruction Tuned) version
            contents=system_prompt
        )
        llm_output = ai_message.content.strip()


        
        # Robust parsing: Ensure the LLM actually used the '|' delimiter
        if '|' in llm_output:
            extracted_memory, refined_query = [part.strip() for part in llm_output.split('|', 1)]
        else:
            print("⚠️ LLaMA 3 formatting error. Bypassing extraction.")
            extracted_memory = "NONE"
            refined_query = user_query
            
        print(f"🧠 Pruned Memory: {extracted_memory}")
        print(f"✨ Refined Query: {refined_query}")
        
    except Exception as e:
        print(f"⚠️ Local LangChain query rewriting failed: {e}")
        extracted_memory = "NONE"
        refined_query = user_query
    
    # --- Step 2: Embed the Refined Natural Language Query ---
    print("🔢 Embedding refined natural language query...")
    query_embed = client.models.embed_content(
        model='gemini-embedding-2-preview',
        contents=refined_query, # Now using the fully constructed sentence!
        config=types.EmbedContentConfig(
            task_type="RETRIEVAL_QUERY",
            output_dimensionality=dimension
        )
    )
    user_vector = query_embed.embeddings[0].values
    
    # --- Step 3: Search ChromaDB ---
    print("🔍 Searching Vector Database for top matches...")
    results = collection.query(
        query_embeddings=[user_vector],
        n_results=5, 
        where={"video_id": video_id}
    )
    
    if not results['metadatas'] or not results['metadatas'][0]:
        final_answer = "I could not find any events matching that description."
        chat_history.append({"question": user_query, "answer": final_answer})
        return final_answer, chat_history
        
    # --- Step 4: Multi-Cluster Logic ---
    chunks = results['metadatas'][0]
    chunks_sorted = sorted(chunks, key=lambda x: x['start_sec'])
    
    clusters = []
    current_cluster = [chunks_sorted[0]]
    
    for chunk in chunks_sorted[1:]:
        last_chunk_end = current_cluster[-1]['end_sec']
        if chunk['start_sec'] - last_chunk_end <= 45.0:
            current_cluster.append(chunk)
        else:
            clusters.append(current_cluster)
            current_cluster = [chunk]
            
    clusters.append(current_cluster)
    print(f"🧩 System identified {len(clusters)} distinct time event(s) across the video.")

    # --- Step 5: Dynamic Trimming & Inline Loading ---
    video_parts = []
    local_clip_paths = []
    
    try:
        for i, cluster in enumerate(clusters):
            cluster_start = min([m['start_sec'] for m in cluster])
            cluster_end = max([m['end_sec'] for m in cluster])
            
            safe_start = max(0.0, cluster_start - 3.0)
            clip_duration = (cluster_end - safe_start) + 3.0 
            
            clip_path = f"temp_cluster_{i}_{int(time.time())}.mp4"
            local_clip_paths.append(clip_path)
            
            print(f"✂️ Extracting Clip {i+1}/{len(clusters)}: {safe_start:.1f}s to {safe_start+clip_duration:.1f}s...")
            extract_video_clip(raw_video_path, safe_start, clip_duration, clip_path)
            
            print(f"📦 Loading Clip {i+1} inline to bypass Cloud Storage...")
            with open(clip_path, "rb") as f:
                video_bytes = f.read()
                
            video_parts.append(
                types.Part.from_bytes(data=video_bytes, mime_type="video/mp4")
            )

        # --- Step 6: Synthesis Continuity (Using Pruned Memory) ---
        print("🧠 Asking Gemini 2.5 Flash to synthesize an answer...")
        
        system_prompt = f"""You are an elite CCTV analysis AI. 
You are being provided with {len(video_parts)} chronologically ordered video clips.
Watch all the clips closely to answer the CURRENT USER QUERY.

INVESTIGATION CONTEXT (Extracted from past turns): 
{extracted_memory}

Use the Investigation Context to resolve any pronouns or track ongoing subjects. 
Return when the event happens, which clip it happens in, and a detailed description."""

        api_contents = video_parts + [system_prompt, f"CURRENT USER QUERY: {user_query}"]

        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=api_contents
        )
        final_answer = response.text

    except Exception as e:
        final_answer = f"Error during final visual analysis: {e}"
        
    finally:
        print("🧹 Cleaning up local temporary files...")
        for l_file in local_clip_paths:
            if os.path.exists(l_file):
                os.remove(l_file)
    
    # Save this turn into the session history before returning
    chat_history.append({"question": user_query, "answer": final_answer})
    return final_answer, chat_history



In [ ]:

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse038_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=15.0)

# --- STEP 2: CHAT ---
QUESTION = "Is there any dangerous activity?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
QUESTION = "Is there any person injured in the incident?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Did the kid get abused?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Did the woman used phone when interact with the kid?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "did the woman make a phone call when interacting with the child?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Did the kid be left unattended?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse042_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Did the kid be left unattended, can you give me the second in the video the kid be left unattended?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse039_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "What happend in the video?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Normal_Videos_for_Event_Recognition/Normal_Videos_576_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Does any approach the red Isuzu car?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Normal_Videos_for_Event_Recognition/Normal_Videos_576_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Did any yellow van or yello truck appear in the video?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Arrest/Arrest040_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Any dangerous activities detected?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Arson/Arson002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "What cause the fire?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-1/Arson/Arson002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Did he comeback after the fire stop?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
session_history = []
TARGET_VIDEO = "Anomaly-Videos-Part-1/Assault/Assault002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "What items are used to attack the victim?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
QUESTION = "So how many of them in total?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
QUESTION = "Is there any other weapon apart from the one you just mention?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
#Normal_Videos_439_x264
# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Normal_Videos_for_Event_Recognition/Normal_Videos_439_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Did anyone steal anything from the store?"

answer = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
session_history = []

TARGET_VIDEO = "Anomaly-Videos-Part-1/Arson/Arson002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "What cause the fire?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

QUESTION = "What color was their shirt?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
session_history = []

TARGET_VIDEO = "Anomaly-Videos-Part-1/Assault/Assault028_x264.mp4" # at night with snow

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Any dangerous activity in the video?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

QUESTION = "How many people involve?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

QUESTION = "Does he still alive?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

print("session")
print(session_history)

In [ ]:
session_history = []

TARGET_VIDEO = "Anomaly-Videos-Part-1/Assault/Assault028_x264.mp4" # at night with snow

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "What is the color of the car the attackers use to run away?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
session_history = []
TARGET_VIDEO = "Anomaly-Videos-Part-1/Assault/Assault004_x264.mp4" # video with low resolution

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "after the attacker leave the screen, did anyone help the victim?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
QUESTION = "How did the attacker flee the scene?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
QUESTION = "Give me the car's characteristic?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
session_history = []
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse002_x264.mp4"

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "Did any accident happends on the road?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
# Point this to your completely raw, unannotated video
session_history = []
TARGET_VIDEO = "Anomaly-Videos-Part-1/Abuse/Abuse002_x264.mp4"

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
#ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=45.0) ## do the ingestion overlapping

# --- STEP 2: CHAT ---
QUESTION = "summary what happen in the video?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
session_history = []

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Normal_Videos_for_Event_Recognition/Normal_Videos_576_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "Summary the video?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:

session_history = []

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-2/Burglary/Burglary039_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "are they allow to enter the property?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)

In [ ]:
session_history = []

# Point this to your completely raw, unannotated video
TARGET_VIDEO = "Anomaly-Videos-Part-2/Burglary/Burglary002_x264.mp4" 

# --- STEP 1: INGESTION ---
# Remember to run this once to index the video! Comment it out after.
ingest_raw_video_direct(TARGET_VIDEO, chunk_duration=30.0)

# --- STEP 2: CHAT ---
QUESTION = "is he allow to enter the property?"

answer, session_history = chat_with_raw_video_direct(QUESTION, TARGET_VIDEO, session_history)

print("\n--- 🤖 FINAL ANSWER ---")
print(answer)